# Embedding Space Math — Exercises

**Chapter 15, Section 02** | [notes.md](notes.md) | [theory.ipynb](theory.ipynb)

10 exercises covering: similarity metrics, analogy arithmetic, positional encodings (sinusoidal + RoPE), isotropy analysis, dimensionality reduction, parameter counting, layer probing, weight tying, and the curse of dimensionality.

Each exercise has:
1. **Problem statement** (markdown)
2. **Scaffold** (code with TODOs)
3. **Solution** (complete implementation)

In [ ]:
import numpy as np
np.random.seed(42)

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("matplotlib not available — plots will be skipped")

---
## Exercise 1: Cosine Similarity Calculator

**Task**: Implement cosine similarity from scratch. Compare two pairs:
1. `cat` vs `dog` (semantically related)
2. `cat` vs `democracy` (semantically unrelated)

Use a **planted** embedding space where related words share a direction.

**Requirements**:
- Write `cosine_similarity(u, v)` without using any library function
- Create embeddings with planted semantic structure
- Print a comparison table of all pairwise similarities
- Verify: $\cos(\mathbf{u}, \mathbf{u}) = 1$ for any non-zero vector

In [ ]:
# === Exercise 1: Scaffold ===

def cosine_similarity(u, v):
    """
    Compute cosine similarity: cos(u,v) = (u·v) / (||u|| ||v||)
    
    TODO: Implement using only np.sum, np.sqrt (no np.linalg)
    """
    pass  # YOUR CODE HERE

# TODO: Create embeddings with planted semantic structure
# Hint: Add a shared "animal" direction to cat and dog
d = 32
animal_dir = np.random.randn(d)
# ...

# TODO: Compute and print pairwise similarities for:
# cat-dog, cat-democracy, dog-democracy, cat-cat
# Print as a table

In [ ]:
# === Exercise 1: Solution ===

def cosine_similarity(u, v):
    """cos(u,v) = (u·v) / (||u|| ||v||)"""
    dot = np.sum(u * v)
    norm_u = np.sqrt(np.sum(u ** 2))
    norm_v = np.sqrt(np.sum(v ** 2))
    if norm_u == 0 or norm_v == 0:
        return 0.0
    return dot / (norm_u * norm_v)

np.random.seed(42)
d = 32

# Planted semantic directions
animal_dir = np.random.randn(d)
animal_dir = animal_dir / np.linalg.norm(animal_dir)
politics_dir = np.random.randn(d)
politics_dir = politics_dir / np.linalg.norm(politics_dir)

# Create embeddings with structure
embeddings = {
    "cat":       np.random.randn(d) * 0.3 + 2.0 * animal_dir,
    "dog":       np.random.randn(d) * 0.3 + 2.0 * animal_dir,
    "democracy": np.random.randn(d) * 0.3 + 2.0 * politics_dir,
    "election":  np.random.randn(d) * 0.3 + 2.0 * politics_dir,
}

# Pairwise similarity table
words = list(embeddings.keys())
print(f"{'':>12}", end="")
for w in words:
    print(f"{w:>12}", end="")
print()

for w1 in words:
    print(f"{w1:>12}", end="")
    for w2 in words:
        sim = cosine_similarity(embeddings[w1], embeddings[w2])
        print(f"{sim:>12.4f}", end="")
    print()

# Verify self-similarity = 1
print(f"\nSelf-similarity check:")
for w in words:
    print(f"  cos({w}, {w}) = {cosine_similarity(embeddings[w], embeddings[w]):.6f}")
print(f"\n✓ cat-dog similarity >> cat-democracy similarity (planted animal direction)")

---
## Exercise 2: Analogy Arithmetic

**Task**: Implement word analogy solving: "A is to B as C is to ?"

$$\vec{v}_? = \vec{v}_C + (\vec{v}_B - \vec{v}_A)$$

Test **5 analogy types** with planted semantic structure:
1. Gender: man → king :: woman → ?
2. Family: man → uncle :: woman → ?
3. Size: small → smaller :: big → ?
4. Tense: walk → walked :: run → ?
5. Geography: Paris → France :: Tokyo → ?

**Requirements**:
- `find_nearest(query_vec, word_dict, exclude)` — cosine-based nearest-neighbor
- Report accuracy (should be 5/5 with well-planted directions)

In [ ]:
# === Exercise 2: Scaffold ===

def find_nearest(query_vec, word_dict, exclude=None):
    """
    Find the word whose embedding is most similar (cosine) to query_vec.
    Exclude words in the 'exclude' set.
    
    TODO: Implement this function
    """
    pass  # YOUR CODE HERE

# TODO: Create word embeddings with 5 planted semantic directions:
# gender_dir, family_dir, size_dir, tense_dir, geo_dir
# Build word vectors as base + direction combinations

# TODO: Test 5 analogies and report accuracy

In [ ]:
# === Exercise 2: Solution ===

def find_nearest(query_vec, word_dict, exclude=None):
    if exclude is None:
        exclude = set()
    best_word, best_sim = None, -np.inf
    for w, v in word_dict.items():
        if w in exclude:
            continue
        sim = cosine_similarity(query_vec, v)
        if sim > best_sim:
            best_sim = sim
            best_word = w
    return best_word, best_sim

np.random.seed(42)
d_an = 64

# Planted semantic directions (orthogonal-ish in high-d space)
def rand_dir(d):
    v = np.random.randn(d)
    return v / np.linalg.norm(v)

gender_dir = rand_dir(d_an)
family_dir = rand_dir(d_an)
size_dir   = rand_dir(d_an)
tense_dir  = rand_dir(d_an)
geo_dir    = rand_dir(d_an)
royalty_dir = rand_dir(d_an)

# Build words: base_noise + direction offsets
w = {}
noise = lambda: np.random.randn(d_an) * 0.2

# Gender + royalty
w["man"]   = noise() + 0 * gender_dir + 0 * royalty_dir
w["woman"] = noise() + 3 * gender_dir + 0 * royalty_dir
w["king"]  = noise() + 0 * gender_dir + 3 * royalty_dir
w["queen"] = noise() + 3 * gender_dir + 3 * royalty_dir

# Family
w["uncle"] = noise() + 0 * gender_dir + 3 * family_dir
w["aunt"]  = noise() + 3 * gender_dir + 3 * family_dir

# Size
w["small"]   = noise() + 0 * size_dir
w["smaller"] = noise() + 3 * size_dir
w["big"]     = noise() + 0 * size_dir + 3 * rand_dir(d_an)  # different base
w["bigger"]  = noise() + 3 * size_dir + 3 * rand_dir(d_an)

# Tense
w["walk"]   = noise() + 0 * tense_dir
w["walked"] = noise() + 3 * tense_dir
w["run"]    = noise() + 0 * tense_dir + 3 * rand_dir(d_an)
w["ran"]    = noise() + 3 * tense_dir + 3 * rand_dir(d_an)

# Geography
w["Paris"]  = noise() + 3 * geo_dir
w["France"] = noise() + 0 * geo_dir + 3 * rand_dir(d_an)
w["Tokyo"]  = noise() + 3 * geo_dir + 3 * rand_dir(d_an)
w["Japan"]  = noise() + 0 * geo_dir + 3 * rand_dir(d_an) + 3 * rand_dir(d_an)

# Analogy tests: A is to B as C is to ?
analogies = [
    ("man",   "king",    "woman",  "queen"),
    ("man",   "uncle",   "woman",  "aunt"),
    ("small", "smaller", "big",    "bigger"),
    ("walk",  "walked",  "run",    "ran"),
    ("Paris", "France",  "Tokyo",  "Japan"),
]

correct = 0
print(f"{'A':<10} {'B':<10} {'C':<10} {'Expected':<10} {'Predicted':<10} {'Cosine':>8} {'✓/✗':>4}")
print("-" * 66)
for a, b, c, expected in analogies:
    offset = w[b] - w[a]
    query = w[c] + offset
    pred, sim = find_nearest(query, w, exclude={a, b, c})
    ok = pred == expected
    correct += ok
    print(f"{a:<10} {b:<10} {c:<10} {expected:<10} {pred:<10} {sim:>8.4f} {'✓' if ok else '✗':>4}")

print(f"\nAccuracy: {correct}/{len(analogies)} = {correct/len(analogies)*100:.0f}%")

---
## Exercise 3: Sinusoidal PE Visualisation

**Task**: Build the sinusoidal positional encoding and analyse its properties.

1. Implement `sinusoidal_pe(max_len, d_model)` → $\text{PE} \in \mathbb{R}^{T \times d}$
2. Plot: (a) heatmap of PE matrix, (b) individual sinusoids, (c) position similarity matrix
3. Verify: $\langle \text{PE}[p+k], \text{PE}[p] \rangle$ is constant for fixed $k$
4. Compute $\|\text{PE}[p]\|_2$ — should be $\sqrt{d/2}$

In [ ]:
# === Exercise 3: Scaffold ===

def sinusoidal_pe(max_len, d_model):
    """
    Build sinusoidal PE matrix.
    PE(pos, 2i)   = sin(pos / 10000^(2i/d))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d))
    
    TODO: Implement. Return PE of shape (max_len, d_model)
    """
    pass  # YOUR CODE HERE

# TODO: Generate PE for max_len=128, d_model=64
# TODO: Verify ||PE[p]||_2 ≈ √(d/2) for all positions
# TODO: Verify inner product PE[p+k]·PE[p] is constant for fixed k
# TODO: Plot heatmap if matplotlib available

In [ ]:
# === Exercise 3: Solution ===

def sinusoidal_pe(max_len, d_model):
    PE = np.zeros((max_len, d_model))
    pos = np.arange(max_len)[:, np.newaxis]
    i = np.arange(0, d_model, 2)[np.newaxis, :]
    div_term = 10000.0 ** (i / d_model)
    angle = pos / div_term
    PE[:, 0::2] = np.sin(angle)
    PE[:, 1::2] = np.cos(angle)
    return PE

max_len, d_model = 128, 64
PE = sinusoidal_pe(max_len, d_model)
print(f"PE shape: {PE.shape}")

# Check 1: Norm ≈ √(d/2)
norms = np.linalg.norm(PE, axis=1)
expected_norm = np.sqrt(d_model / 2)
print(f"\n||PE[p]||_2:  mean={norms.mean():.4f}, expected √(d/2)={expected_norm:.4f}")
print(f"  All within 1% of expected: {np.allclose(norms, expected_norm, rtol=0.01)}")

# Check 2: Relative position property
print(f"\nRelative position property (PE[p+k]·PE[p] constant for fixed k):")
for k in [1, 3, 10, 25]:
    dots = [np.dot(PE[p], PE[p + k]) for p in range(0, max_len - k)]
    print(f"  k={k:>3}: mean={np.mean(dots):>8.4f}, std={np.std(dots):>10.6f} "
          f"{'✓ constant' if np.std(dots) < 0.01 else '≈ constant'}")

# Visualise
if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    ax = axes[0]
    ax.imshow(PE[:64, :32].T, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xlabel('Position'); ax.set_ylabel('Dimension')
    ax.set_title('Sinusoidal PE Heatmap')
    
    ax = axes[1]
    for dim in [0, 1, 6, 16, 30, 62]:
        ax.plot(PE[:, dim], label=f'd={dim}', alpha=0.7)
    ax.set_xlabel('Position'); ax.set_ylabel('PE value')
    ax.set_title('Individual Sinusoids'); ax.legend(fontsize=7)
    
    ax = axes[2]
    PE_n = PE / np.linalg.norm(PE, axis=1, keepdims=True)
    sim_mat = PE_n[:50] @ PE_n[:50].T
    ax.imshow(sim_mat, cmap='viridis')
    ax.set_title('Position Cosine Similarity')
    ax.set_xlabel('Position'); ax.set_ylabel('Position')
    
    plt.tight_layout(); plt.show()
else:
    print("matplotlib not available — skipping plots")

---
## Exercise 4: Embedding Isotropy Analysis

**Task**: Measure and fix embedding space isotropy.

1. Generate **isotropic** embeddings (random Gaussian)
2. Generate **anisotropic** embeddings (shared mean + small noise)
3. Measure average pairwise cosine similarity (should be ≈0 for isotropic)
4. Fix anisotropy via **mean-centering** and **ZCA whitening**
5. Show before/after metrics

In [ ]:
# === Exercise 4: Scaffold ===

def avg_pairwise_cosine(embeddings):
    """
    Compute average pairwise cosine similarity.
    
    TODO: Loop over all pairs (i < j), compute cosine_similarity,
    return mean and std.
    """
    pass  # YOUR CODE HERE

def mean_center(X):
    """TODO: Subtract the mean vector from all embeddings."""
    pass  # YOUR CODE HERE

def zca_whiten(X):
    """TODO: ZCA whitening: (X - μ) @ Σ^{-1/2}"""
    pass  # YOUR CODE HERE

# TODO: Generate isotropic (np.random.randn) and anisotropic embeddings
# TODO: Apply fixes and compare

In [ ]:
# === Exercise 4: Solution ===

def avg_pairwise_cosine(embeddings):
    n = len(embeddings)
    sims = []
    for i in range(n):
        for j in range(i + 1, n):
            sims.append(cosine_similarity(embeddings[i], embeddings[j]))
    return np.mean(sims), np.std(sims)

def mean_center(X):
    return X - np.mean(X, axis=0)

def zca_whiten(X):
    centered = X - np.mean(X, axis=0)
    cov = np.cov(centered, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)
    eigvals = np.maximum(eigvals, 1e-8)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(eigvals))
    W = eigvecs @ D_inv_sqrt @ eigvecs.T
    return centered @ W

np.random.seed(42)
n_tok, d_iso = 150, 64

# Isotropic: random Gaussian
iso = np.random.randn(n_tok, d_iso)

# Anisotropic: strong shared direction + small perturbation
shared_dir = np.random.randn(d_iso) * 5.0
aniso = shared_dir + np.random.randn(n_tok, d_iso) * 0.3

# Measure
avg_iso, std_iso = avg_pairwise_cosine(iso)
avg_aniso, std_aniso = avg_pairwise_cosine(aniso)

# Fix
aniso_centered = mean_center(aniso)
aniso_whitened = zca_whiten(aniso)
avg_cent, _ = avg_pairwise_cosine(aniso_centered)
avg_whit, _ = avg_pairwise_cosine(aniso_whitened)

print("=== Isotropy Analysis ===")
print(f"{'Type':<20} {'Avg Cosine':>12} {'Std':>8} {'Diagnosis':>15}")
print("-" * 59)
print(f"{'Isotropic (ref)':<20} {avg_iso:>12.4f} {std_iso:>8.4f} {'✓ Good':>15}")
print(f"{'Anisotropic':<20} {avg_aniso:>12.4f} {std_aniso:>8.4f} {'✗ Degenerate':>15}")
print(f"{'Mean-centered':<20} {avg_cent:>12.4f} {'':>8} {'✓ Fixed':>15}")
print(f"{'ZCA Whitened':<20} {avg_whit:>12.4f} {'':>8} {'✓ Fixed':>15}")

---
## Exercise 5: RoPE Implementation

**Task**: Implement Rotary Position Embedding from scratch.

1. Compute frequencies: $\theta_i = 10000^{-2i/d}$
2. Apply 2×2 rotation to each dimension pair: $R_i(m) = \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix}$
3. Verify: $\text{RoPE}(q, m)^\top \text{RoPE}(k, n)$ depends only on $m - n$, not on absolute positions

In [ ]:
# === Exercise 5: Scaffold ===

def rope_frequencies(d, base=10000.0):
    """TODO: Return θ_i = 1/base^(2i/d) for i = 0,...,d/2-1"""
    pass  # YOUR CODE HERE

def apply_rope(x, position, theta):
    """
    Apply RoPE to vector x at given position.
    For each pair (x[2i], x[2i+1]), apply 2×2 rotation by angle pos·θ_i.
    
    TODO: Implement
    """
    pass  # YOUR CODE HERE

# TODO: Verify relative position property
# For fixed offset k, check that dot(RoPE(q,m), RoPE(k,n)) is constant
# when m-n = k, regardless of absolute positions m, n

In [ ]:
# === Exercise 5: Solution ===

def rope_frequencies(d, base=10000.0):
    i = np.arange(0, d, 2)
    return 1.0 / (base ** (i / d))

def apply_rope(x, position, theta):
    d = len(x)
    x_rot = np.copy(x)
    for i in range(d // 2):
        angle = position * theta[i]
        c, s = np.cos(angle), np.sin(angle)
        x0, x1 = x[2*i], x[2*i+1]
        x_rot[2*i]     = c * x0 - s * x1
        x_rot[2*i + 1] = s * x0 + c * x1
    return x_rot

np.random.seed(42)
d_r = 16
theta = rope_frequencies(d_r)
q = np.random.randn(d_r)
k = np.random.randn(d_r)

print(f"RoPE frequencies: {np.round(theta, 6)}\n")

# Verify relative position property
print("Relative position verification:")
print(f"{'(m, n)':<14} {'offset':>7} {'dot(q_m, k_n)':>15}")
print("-" * 40)

for offset in [0, 3, 7, 15]:
    dots = []
    for base_pos in [0, 10, 50, 100]:
        m = base_pos + offset
        n = base_pos
        q_m = apply_rope(q, m, theta)
        k_n = apply_rope(k, n, theta)
        dot_val = np.dot(q_m, k_n)
        dots.append(dot_val)
        if base_pos <= 10:
            print(f"({m:>3}, {n:>3})      {offset:>7}   {dot_val:>15.6f}")
    # Check constancy
    std = np.std(dots)
    print(f"  → offset={offset}: std across positions = {std:.2e} {'✓ constant' if std < 1e-10 else ''}\n")

---
## Exercise 6: Dimensionality Reduction

**Task**: Implement PCA from scratch and apply to clustered embeddings.

1. Create 500 embeddings in $\mathbb{R}^{64}$ with 5 semantic clusters
2. Implement PCA via eigendecomposition of the covariance matrix
3. Project to 2D and verify clusters are visible
4. Plot the scree plot (cumulative explained variance)
5. Report: how many PCs capture 90% of variance?

In [ ]:
# === Exercise 6: Scaffold ===

def pca_from_scratch(X, n_components=2):
    """
    PCA via eigendecomposition.
    
    1. Center X
    2. Compute covariance matrix
    3. Eigendecompose
    4. Project onto top-k eigenvectors
    
    TODO: Implement. Return (Z, eigenvalues, explained_var_ratio)
    """
    pass  # YOUR CODE HERE

# TODO: Create 5 clusters of 100 embeddings each in R^64
# TODO: Run PCA, report explained variance, find 90% threshold
# TODO: Plot scatter + scree if matplotlib available

In [ ]:
# === Exercise 6: Solution ===

def pca_from_scratch(X, n_components=2):
    mu = np.mean(X, axis=0)
    X_c = X - mu
    cov = (X_c.T @ X_c) / (X_c.shape[0] - 1)
    eigvals, eigvecs = np.linalg.eigh(cov)
    idx = np.argsort(eigvals)[::-1]
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]
    W = eigvecs[:, :n_components]
    Z = X_c @ W
    explained = eigvals / np.sum(eigvals)
    return Z, eigvals, explained

np.random.seed(42)
n_per = 100
d_pca = 64
cluster_names = ["animals", "colors", "verbs", "numbers", "places"]

all_X = []
all_labels = []
for i, name in enumerate(cluster_names):
    center = np.random.randn(d_pca) * 3.0
    cluster = center + np.random.randn(n_per, d_pca) * 0.5
    all_X.append(cluster)
    all_labels.extend([name] * n_per)
X_all = np.vstack(all_X)

Z, eigvals, explained = pca_from_scratch(X_all, n_components=2)

print(f"=== PCA Results ({X_all.shape[0]} points, {d_pca}D → 2D) ===")
print(f"PC1: {explained[0]*100:.1f}%  |  PC2: {explained[1]*100:.1f}%")
print(f"Top 2 PCs: {sum(explained[:2])*100:.1f}%\n")

cum = np.cumsum(explained)
for target in [0.5, 0.75, 0.9, 0.95, 0.99]:
    k = np.searchsorted(cum, target) + 1
    print(f"  {target*100:.0f}% variance captured by {k} PCs (of {d_pca})")

# Cluster centroids in 2D
print(f"\n2D Centroids:")
for i, name in enumerate(cluster_names):
    c = Z[i*n_per:(i+1)*n_per].mean(axis=0)
    print(f"  {name:<10}: ({c[0]:>7.3f}, {c[1]:>7.3f})")

if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    colors_map = ['red', 'blue', 'green', 'orange', 'purple']
    for i, name in enumerate(cluster_names):
        pts = Z[i*n_per:(i+1)*n_per]
        ax1.scatter(pts[:, 0], pts[:, 1], c=colors_map[i], label=name, alpha=0.5, s=20)
    ax1.set_xlabel(f"PC1 ({explained[0]*100:.1f}%)")
    ax1.set_ylabel(f"PC2 ({explained[1]*100:.1f}%)")
    ax1.set_title("PCA of Clustered Embeddings"); ax1.legend(); ax1.grid(alpha=0.3)
    
    ax2.bar(range(1, 21), explained[:20]*100, color='steelblue', alpha=0.7)
    ax2.plot(range(1, 21), cum[:20]*100, 'r-o', markersize=3, label='Cumulative')
    ax2.axhline(y=90, color='gray', linestyle='--', alpha=0.5, label='90%')
    ax2.set_xlabel("PC"); ax2.set_ylabel("Variance (%)"); ax2.set_title("Scree Plot")
    ax2.legend(); ax2.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

---
## Exercise 7: Parameter Counting

**Task**: For 7 real LLM architectures, compute:
1. Embedding parameters ($N \times d$)
2. Embedding parameters as % of total model parameters
3. If weight-tied, what are the savings?

Display results in a formatted table. Discuss the trend: as models scale, embedding % shrinks.

In [ ]:
# === Exercise 7: Scaffold ===

# TODO: Define a list of models with (name, vocab_size, d_model, total_params, tied?)
# Models: GPT-2 Small, GPT-2 XL, GPT-3, BERT-base, BERT-large, LLaMA-7B, LLaMA-65B

# TODO: For each model compute:
#   embed_params = vocab_size * d_model
#   embed_pct = embed_params / total_params * 100
#   tied_savings = embed_params  (if applicable)

# TODO: Print formatted table

In [ ]:
# === Exercise 7: Solution ===

#                   name            vocab    d_model  total_params   tied?
models = [
    ("GPT-2 Small",    50257,    768,   124_000_000, True),
    ("GPT-2 XL",       50257,   1600,  1_500_000_000, True),
    ("GPT-3 175B",     50257,  12288, 175_000_000_000, False),
    ("BERT-base",      30522,    768,   110_000_000, True),
    ("BERT-large",     30522,   1024,   340_000_000, True),
    ("LLaMA-7B",       32000,   4096,  7_000_000_000, False),
    ("LLaMA-65B",      32000,   8192, 65_000_000_000, False),
]

print(f"{'Model':<16} {'Vocab':>8} {'d':>6} {'Embed Params':>14} {'Total':>13} {'Embed%':>7} {'Tied':>5} {'Savings':>12}")
print("-" * 86)
for name, vocab, d, total, tied in models:
    emb = vocab * d
    pct = emb / total * 100
    savings = emb if tied else 0
    print(f"{name:<16} {vocab:>8,} {d:>6} {emb:>14,} {total:>13,} {pct:>6.2f}% {'Yes' if tied else 'No':>5} {savings:>12,}")

print(f"\n--- Trend Analysis ---")
print(f"Small models: embedding is a large fraction (>10%) of total parameters.")
print(f"Large models: embedding fraction shrinks (<1%) as depth dominates.")
print(f"Weight tying saves N×d params — significant for small models, negligible for large.")

---
## Exercise 8: Layer Probing

**Task**: Simulate a 12-layer Transformer and probe how embeddings change across layers.

1. Pass embeddings through 12 simplified layers (attention + FFN + residual)
2. At layers 1, 4, 8, 12: measure intra-cluster vs inter-cluster cosine similarity
3. Show that deeper layers produce more separated representations
4. Compute cosine similarity between layer-0 and layer-L embeddings (signal persistence)

In [ ]:
# === Exercise 8: Scaffold ===

# TODO: Create 3 clusters of 5 tokens each in R^32 (simulating a small vocab)
# TODO: Build a 12-layer simplified Transformer pipeline
# TODO: At each layer:
#   1. Compute Q = X @ W_Q, K = X @ W_K, V = X @ W_V
#   2. Apply scaled dot-product attention
#   3. Add residual: X = X + attn_output * scale
#   4. Apply simple FFN + residual
# TODO: At layers {1, 4, 8, 12}, measure:
#   - intra-cluster avg cosine similarity
#   - inter-cluster avg cosine similarity
#   - cos(embedding_layer0, embedding_layerL)

In [ ]:
# === Exercise 8: Solution ===

np.random.seed(42)
d_lp = 32
d_ff_lp = 64
n_layers = 12
tokens_per_cluster = 5
n_clusters = 3
T_lp = n_clusters * tokens_per_cluster

# Create clustered initial embeddings
cluster_centers = [np.random.randn(d_lp) * 2.0 for _ in range(n_clusters)]
X_init = np.vstack([
    center + np.random.randn(tokens_per_cluster, d_lp) * 0.3
    for center in cluster_centers
])
X_layer0 = X_init.copy()

def simple_attn_layer(X, d, scale_factor=0.1):
    W_Q = np.random.randn(d, d) * (1.0/np.sqrt(d))
    W_K = np.random.randn(d, d) * (1.0/np.sqrt(d))
    W_V = np.random.randn(d, d) * (1.0/np.sqrt(d))
    Q, K, V = X @ W_Q, X @ W_K, X @ W_V
    scores = Q @ K.T / np.sqrt(d)
    scores -= np.max(scores, axis=-1, keepdims=True)
    attn = np.exp(scores) / np.sum(np.exp(scores), axis=-1, keepdims=True)
    out = attn @ V
    return X + out * scale_factor

def simple_ffn_layer(X, d, d_ff, scale_factor=0.1):
    W1 = np.random.randn(d, d_ff) * (1.0/np.sqrt(d))
    W2 = np.random.randn(d_ff, d) * (1.0/np.sqrt(d_ff))
    hidden = np.maximum(0, X @ W1)
    out = hidden @ W2
    return X + out * scale_factor

# Run through layers, snapshot at probe points
probe_layers = [1, 4, 8, 12]
snapshots = {0: X_init.copy()}
X_cur = X_init.copy()

for l in range(1, n_layers + 1):
    X_cur = simple_attn_layer(X_cur, d_lp)
    X_cur = simple_ffn_layer(X_cur, d_lp, d_ff_lp)
    if l in probe_layers:
        snapshots[l] = X_cur.copy()

# Measure intra- and inter-cluster similarity at each probe point
def cluster_similarities(X, tpc, nc):
    intra_sims, inter_sims = [], []
    for ci in range(nc):
        for i in range(tpc):
            for j in range(i+1, tpc):
                idx_i = ci * tpc + i
                idx_j = ci * tpc + j
                intra_sims.append(cosine_similarity(X[idx_i], X[idx_j]))
    for ci in range(nc):
        for cj in range(ci+1, nc):
            for i in range(tpc):
                for j in range(tpc):
                    idx_i = ci * tpc + i
                    idx_j = cj * tpc + j
                    inter_sims.append(cosine_similarity(X[idx_i], X[idx_j]))
    return np.mean(intra_sims), np.mean(inter_sims)

print("=== Layer Probing: Cluster Separation ===\n")
print(f"{'Layer':>6} {'Intra-cluster cos':>20} {'Inter-cluster cos':>20} {'Separation':>12} {'cos(L0,L)':>12}")
print("-" * 74)
for l in [0] + probe_layers:
    X_snap = snapshots[l]
    intra, inter = cluster_similarities(X_snap, tokens_per_cluster, n_clusters)
    # cos with initial embedding
    cos_init = np.mean([cosine_similarity(X_layer0[i], X_snap[i]) for i in range(T_lp)])
    sep = intra - inter
    print(f"{l:>6} {intra:>20.4f} {inter:>20.4f} {sep:>12.4f} {cos_init:>12.4f}")

print(f"\n→ Intra-cluster similarity stays high; inter-cluster drops (better separation).")
print(f"  cos(L0, L) decreases with depth but stays > 0 (residual stream preserves signal).")

---
## Exercise 9: Weight Tying Analysis

**Task**: Compare tied vs untied embedding approaches.

1. Create a tiny language model (vocab=200, d=64, 1 hidden layer)
2. **Untied**: separate $E_{in}$ and $E_{out}$
3. **Tied**: $E_{out} = E_{in}^\top$
4. Simulate training (simplified forward + loss)
5. Compare: parameter count, logit distribution, embedding-output alignment

In [ ]:
# === Exercise 9: Scaffold ===

# TODO: Build a tiny LM with:
#   - E_in: (V, d) input embedding
#   - W_h: (d, d) hidden layer weight
#   - E_out: (d, V) output projection  [separate in untied; E_in.T in tied]
#
# Forward: logits = ReLU(E_in[token] @ W_h) @ E_out
# Loss: cross-entropy with target token
#
# Compare:
#   1. Parameter counts (tied vs untied)
#   2. Logit statistics
#   3. cos(E_in[i], E_out[:, i]) — alignment between input and output embeddings

In [ ]:
# === Exercise 9: Solution ===

np.random.seed(42)
V_wt, d_wt = 200, 64

# Init
E_in = np.random.randn(V_wt, d_wt) * (1.0 / np.sqrt(d_wt))
W_h  = np.random.randn(d_wt, d_wt) * (1.0 / np.sqrt(d_wt))

# Untied: separate output projection
E_out_untied = np.random.randn(d_wt, V_wt) * (1.0 / np.sqrt(d_wt))

# Tied: E_out = E_in.T
E_out_tied = E_in.T

# Forward pass for a random input token
token_id = 42
e_in = E_in[token_id]
hidden = np.maximum(0, e_in @ W_h)  # ReLU

logits_untied = hidden @ E_out_untied
logits_tied   = hidden @ E_out_tied

def softmax(x):
    x_s = x - np.max(x)
    e = np.exp(x_s)
    return e / e.sum()

probs_untied = softmax(logits_untied)
probs_tied   = softmax(logits_tied)

# Parameter counts
params_tied   = V_wt * d_wt + d_wt * d_wt  # E_in + W_h (E_out is free)
params_untied = V_wt * d_wt + d_wt * d_wt + d_wt * V_wt  # E_in + W_h + E_out

print("=== Weight Tying Comparison ===\n")
print(f"{'Metric':<30} {'Untied':>14} {'Tied':>14}")
print("-" * 62)
print(f"{'Parameters':<30} {params_untied:>14,} {params_tied:>14,}")
print(f"{'Savings':<30} {'':>14} {params_untied - params_tied:>14,}")
print(f"{'Logit mean':<30} {logits_untied.mean():>14.4f} {logits_tied.mean():>14.4f}")
print(f"{'Logit std':<30} {logits_untied.std():>14.4f} {logits_tied.std():>14.4f}")
print(f"{'Top-1 prob':<30} {probs_untied.max():>14.4f} {probs_tied.max():>14.4f}")
print(f"{'Entropy (nats)':<30} {-np.sum(probs_untied * np.log(probs_untied+1e-10)):>14.4f} "
      f"{-np.sum(probs_tied * np.log(probs_tied+1e-10)):>14.4f}")

# Alignment: E_in[i] vs E_out[:, i]
print(f"\nInput-Output embedding alignment (cosine):")
cos_untied = [cosine_similarity(E_in[i], E_out_untied[:, i]) for i in range(V_wt)]
cos_tied   = [cosine_similarity(E_in[i], E_out_tied[:, i]) for i in range(V_wt)]
print(f"  Untied: mean cos = {np.mean(cos_untied):.4f} (random — no alignment)")
print(f"  Tied:   mean cos = {np.mean(cos_tied):.4f} (perfect — same matrix)")
print(f"\n→ Tied embeddings guarantee input-output alignment by construction.")

---
## Exercise 10: Curse of Dimensionality Demo

**Task**: Demonstrate how pairwise distances concentrate as dimensionality increases.

1. For $d \in \{2, 10, 50, 100, 500, 1000, 4096\}$:
   - Generate 200 random unit vectors in $\mathbb{R}^d$
   - Compute all pairwise Euclidean distances
   - Record: mean, std, min, max, ratio = (max−min)/mean
2. Plot the concentration phenomenon
3. Show: the coefficient of variation (std/mean) shrinks with $d$
4. Explain: why this matters for nearest-neighbor search in embedding spaces

In [ ]:
# === Exercise 10: Scaffold ===

# TODO: For each d in [2, 10, 50, 100, 500, 1000, 4096]:
#   1. Generate 200 random unit vectors
#   2. Compute ALL pairwise Euclidean distances
#   3. Record mean, std, min, max, (max-min)/mean, std/mean
#
# TODO: Print table and plot if matplotlib available
# TODO: Explain why CV shrinks and what it means for search

In [ ]:
# === Exercise 10: Solution ===

np.random.seed(42)
dims = [2, 10, 50, 100, 500, 1000, 4096]
n_vecs = 200

print("=== Curse of Dimensionality: Distance Concentration ===\n")
print(f"{'Dim d':>6} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8} {'(Max-Min)/Mean':>16} {'CV (Std/Mean)':>14}")
print("-" * 74)

results_cod = []
for d in dims:
    # Random unit vectors
    X = np.random.randn(n_vecs, d)
    X = X / np.linalg.norm(X, axis=1, keepdims=True)
    
    # Pairwise distances (upper triangle only)
    dists = []
    for i in range(n_vecs):
        for j in range(i+1, n_vecs):
            dists.append(np.linalg.norm(X[i] - X[j]))
    dists = np.array(dists)
    
    mn = dists.mean()
    sd = dists.std()
    mi = dists.min()
    mx = dists.max()
    ratio = (mx - mi) / mn
    cv = sd / mn
    results_cod.append((d, mn, sd, mi, mx, ratio, cv))
    
    print(f"{d:>6} {mn:>8.4f} {sd:>8.4f} {mi:>8.4f} {mx:>8.4f} {ratio:>16.4f} {cv:>14.4f}")

print(f"\n→ As d ↑:")
print(f"  • Mean distance → √2 (CLT: sum of d terms, each ≈ 2/d)")
print(f"  • CV (std/mean) → 0 (all distances become indistinguishable)")
print(f"  • (Max-Min)/Mean → 0 (nearest ≈ farthest neighbor)")
print(f"\n  This is why brute-force nearest-neighbor struggles in high-d.")
print(f"  LLMs mitigate this by learning structured embeddings (not random).")

if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    ds = [r[0] for r in results_cod]
    cvs = [r[6] for r in results_cod]
    ratios = [r[5] for r in results_cod]
    
    ax1.plot(ds, cvs, 'bo-', linewidth=2)
    ax1.set_xlabel('Dimensionality d'); ax1.set_ylabel('Coefficient of Variation')
    ax1.set_title('Distance Concentration'); ax1.set_xscale('log'); ax1.grid(alpha=0.3)
    
    ax2.plot(ds, ratios, 'ro-', linewidth=2)
    ax2.set_xlabel('Dimensionality d'); ax2.set_ylabel('(Max-Min)/Mean')
    ax2.set_title('Distance Contrast Ratio'); ax2.set_xscale('log'); ax2.grid(alpha=0.3)
    
    plt.tight_layout(); plt.show()

---
## Summary

| Exercise | Topic | Key Math |
|----------|-------|----------|
| 1 | Cosine Similarity | $\cos(\mathbf{u},\mathbf{v}) = \frac{\mathbf{u}\cdot\mathbf{v}}{\|\mathbf{u}\|\|\mathbf{v}\|}$ |
| 2 | Analogy Arithmetic | $\vec{?} = \vec{C} + (\vec{B} - \vec{A})$ |
| 3 | Sinusoidal PE | $\text{PE}(pos, 2i) = \sin(pos / 10000^{2i/d})$ |
| 4 | Isotropy Analysis | Avg pairwise cosine → 0 is isotropic |
| 5 | RoPE | $R_i(m)^\top R_i(n) = R_i(n-m)$ |
| 6 | PCA | Top-$k$ eigenvectors of covariance matrix |
| 7 | Parameter Counting | Embed params = $N \times d$ |
| 8 | Layer Probing | Cluster separation increases with depth |
| 9 | Weight Tying | $E_{out} = E_{in}^\top$, saves $N \times d$ params |
| 10 | Curse of Dimensionality | CV $= \sigma/\mu \to 0$ as $d \to \infty$ |

**Next**: [../03-Attention-Math/](../03-Attention-Math/) for attention mechanism mathematics.